# Exploratory Data Analysis (EDA) - Prediksi Churn Pelanggan

**Dataset:** Sales and Marketing Customer Dataset  
**Tujuan:** Memahami karakteristik data, mengidentifikasi missing value, melihat distribusi target, dan menganalisis korelasi antar fitur numerik.

## 1. Load Data & Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Biar visualisasi lebih rapi
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load dataset
df = pd.read_csv('../dataset/Sales - Marketing customer dataset.csv')
print(f'Dataset berhasil dimuat dengan {df.shape[0]} baris dan {df.shape[1]} kolom.')

## 2. Menampilkan 5 Baris Pertama, Informasi Dataset, dan Statistik Deskriptif

In [ ]:
# Menampilkan 5 baris pertama
df.head()

In [ ]:
# Informasi dataset (tipe data, jumlah non-null, memory usage)
df.info()

In [ ]:
# Statistik deskriptif untuk fitur numerik
df.describe()

In [ ]:
# Statistik deskriptif untuk fitur kategorikal
df.describe(include='object')

**Insight:**
- Dataset terdiri dari 15.000 baris dan 30 kolom.
- Terdapat campuran tipe data: 10 kolom float64, 10 kolom int64, dan 10 kolom object.
- Beberapa kolom memiliki jumlah non-null yang lebih rendah dari total baris, menandakan adanya missing value.
- Dari statistik deskriptif, terlihat kolom `age` memiliki nilai minimum -4 yang kemungkinan merupakan data anomali/error.

## 3. Persentase Missing Value & Visualisasi

In [ ]:
# Menghitung jumlah dan persentase missing value tiap kolom
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100

# Gabungkan jadi satu DataFrame biar gampang dibaca
missing_df = pd.DataFrame({
    'Jumlah Missing': missing_count,
    'Persentase (%)': missing_pct
}).sort_values(by='Persentase (%)', ascending=False)

print('=== Persentase Missing Value Semua Kolom ===')
print(missing_df)

In [ ]:
# Filter hanya kolom yang punya missing value > 0
missing_only = missing_df[missing_df['Jumlah Missing'] > 0].copy()
missing_only = missing_only.reset_index().rename(columns={'index': 'Kolom'})

# Visualisasi diagram batang
plt.figure(figsize=(10, 5))
bars = plt.barh(missing_only['Kolom'], missing_only['Persentase (%)'], color=sns.color_palette('Reds_r', len(missing_only)))

# Tambahin label persentase di ujung bar
for bar, pct in zip(bars, missing_only['Persentase (%)']):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{pct:.2f}%', va='center', fontsize=10)

plt.xlabel('Persentase Missing (%)')
plt.ylabel('Kolom')
plt.title('Persentase Missing Value per Kolom', fontsize=13, fontweight='bold')
plt.xlim(0, max(missing_only['Persentase (%)']) + 8)
plt.tight_layout()
plt.show()

**Insight:**
- Terdapat 5 kolom yang memiliki missing value dari total 30 kolom.
- `coupon_code` memiliki missing value tertinggi (~40.89%), yang wajar karena tidak semua pelanggan menggunakan kupon.
- Kolom `age` (~8%), `total_spent` (~7%), `gender` (~4.92%), dan `satisfaction_score` (~4.68%) juga memiliki data kosong yang perlu ditangani di tahap preprocessing.
- Strategi penanganan missing value akan dilakukan di tahap Data Preparation.

## 4. Distribusi Variabel Target (Churn)

In [ ]:
# Cek distribusi kelas target
print('Distribusi kelas target (churn):')
print(df['churn'].value_counts())
print()
print('Persentase masing-masing kelas:')
print(df['churn'].value_counts(normalize=True).apply(lambda x: f'{x*100:.2f}%'))

In [ ]:
# Visualisasi distribusi target dengan countplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
ax1 = axes[0]
churn_counts = df['churn'].value_counts()
bars = ax1.bar(churn_counts.index.astype(str), churn_counts.values,
               color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)

for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 100,
             f'{int(height)}\n({height/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontsize=11)

ax1.set_xlabel('Status Churn')
ax1.set_ylabel('Jumlah Pelanggan')
ax1.set_title('Distribusi Churn (Bar Chart)', fontsize=12, fontweight='bold')
ax1.set_xticks([0, 1])
ax1.set_xticklabels(['0 (Bertahan)', '1 (Churn)'])

# Pie chart
ax2 = axes[1]
ax2.pie(churn_counts.values, labels=['Bertahan (0)', 'Churn (1)'],
        autopct='%1.1f%%', startangle=90,
        colors=['#2ecc71', '#e74c3c'],
        explode=(0, 0.05), shadow=True)
ax2.set_title('Distribusi Churn (Pie Chart)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

**Insight:**
- Dataset menunjukkan ketidakseimbangan kelas (class imbalance) yang cukup signifikan.
- Pelanggan yang bertahan (0) mendominasi sebesar ~84.68% (12.702 data).
- Pelanggan yang churn (1) hanya ~15.32% (2.298 data).
- Ketidakseimbangan ini perlu diperhatikan saat pemodelan nanti, bisa menggunakan teknik seperti SMOTE, class_weight, atau stratified sampling agar model tidak bias ke kelas mayoritas.

## 5. Heatmap Korelasi Fitur Numerik

In [ ]:
# Ambil fitur numerik saja (exclude customer_id karena itu cuma identifier)
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
numeric_cols.remove('customer_id')  # buang ID, bukan fitur prediktif

# Hitung matriks korelasi
corr_matrix = df[numeric_cols].corr()

# Visualisasi heatmap
plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # mask segitiga atas biar ga redundan

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            square=True, cbar_kws={'shrink': 0.8})

plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi masing-masing fitur terhadap target churn, diurutkan
corr_with_churn = df[numeric_cols].corr()['churn'].drop('churn').sort_values(ascending=False)

print('=== Korelasi Fitur Numerik terhadap Churn ===')
print(corr_with_churn.to_string())

**Insight:**
- Secara umum, mayoritas fitur numerik memiliki korelasi linear yang lemah satu sama lain (mendekati 0).
- Terhadap variabel target `churn`, beberapa hubungan yang menonjol:
  - `satisfaction_score` memiliki korelasi negatif paling kuat (~-0.30), artinya semakin rendah kepuasan pelanggan, semakin besar kemungkinan churn.
  - `support_tickets` berkorelasi positif (~0.13), menunjukkan pelanggan yang sering komplain lebih rentan churn.
  - `total_spent` berkorelasi negatif (~-0.16), pelanggan yang banyak belanja cenderung lebih loyal.
- Tidak ada multikolinearitas yang parah antar fitur prediktor, sehingga sebagian besar fitur bisa digunakan bersamaan tanpa masalah redundansi.

---
## Kesimpulan EDA

1. Dataset memiliki 15.000 baris dan 30 kolom dengan variasi tipe data (numerik, kategorikal, biner, ordinal).
2. Terdapat 5 kolom dengan missing value, yang terbesar adalah `coupon_code` (40.89%).
3. Distribusi target menunjukkan class imbalance: 84.68% bertahan vs 15.32% churn.
4. Fitur `satisfaction_score`, `support_tickets`, dan `total_spent` memiliki korelasi paling signifikan terhadap churn.
5. Langkah selanjutnya: penanganan missing value, encoding fitur kategorikal, dan penanganan class imbalance sebelum pemodelan.